In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import viewer
import dianne
import pandas as pd
import matplotlib.colors as mcolors
dianne.setNotebookWidth(90)
import os
def createH2(slide, mpath=None):
    df_temp = pd.read_csv(f'{mpath}/{slide}.matrix-H.csv', header=None)
    df_temp.iloc[:2,:2] *= 0.25 / 0.2208187960959237
    sname = f'{mpath}/{slide}.matrix-H-2.csv'
    if not os.path.isfile(sname):
        df_temp.to_csv(sname, index=False, header=False)
    return
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)

In [ ]:
stqPath = '/projects/activities/kappsen-tmc/USERS/domans/results-placenta-membranes/'
xePath = '/projects/activities/kappsen-tmc/USERS/domans/dev-MIQ-sequential/run-xenium-ranger'
mpath = '/projects/activities/kappsen-tmc/USERS/domans/Fetal-membranes-analysis/SenReg-results'
sotPath = '/projects/activities/kappsen-tmc/USERS/domans/dev-MIQ-sequential/spatial-omics-tools/segmentation/mesmer/results'

samples = ['MC_PLACM_0003_5k',
           'MC_PLACM_0011_5k']
F = 2
fname = f'features/false-{F}-ctranspath_features.tsv.gz'
patch_size = 8 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
ts, mpp, tile_size = dianne.loadSTQParams(stqPath + samples[0], F)
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, stqPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)
sizes = {s: ads[s].shape[0] for s in samples}
print(f'Prepared {patchesCDFs.shape[0]} patches')

runfn = dianne.makeRunFn(patchCoordinates, ads, samples, qs, ts, mpp, tile_size=tile_size, patch_size=patch_size, PCMA_alpha=0.8, alpha_img=0.5, multiplier=2, erode=True)
savefn = dianne.makeSaveFn(patchCoordinates, ads, samples, qs, ts, mpp, PCMA_alpha=0.8, tile_size=tile_size, patch_size=patch_size, body_overlap=0.25, classifierPaths=classifierPaths)
loadfn = dianne.makeLoadFn(classifierPaths)
listfn = dianne.makeListFn(classifierPaths)

xenium_bundle_paths = {'MC_PLACM_0011_5k': f'{xePath}/region1-PLACM-11-reseg-v7',
                       'MC_PLACM_0003_5k': f'{xePath}/region2-PLACM-3-reseg-v7'}

for sample in samples:
    createH2(sample + '.he', mpath)
xenium_to_he_matrices = {sample: f'{mpath}/{sample}.he.matrix-H-2.csv' for sample in samples}

mif_images = {'MC_PLACM_0011_5k': f'{sotPath}/XE5k-PLACM-0011-v7/adjusted-all-channels.ome.tif',
              'MC_PLACM_0003_5k': f'{sotPath}/XE5k-PLACM-0003-v7/adjusted-all-channels.ome.tif'}
identity_matrices = {sample: './identity-matrix.csv' for sample in samples}

drawings = viewer.create_viewer(samples, mif_images, height="800px", run_inference_fn=runfn, sample_sizes=sizes,
                                xenium_mpp=0.2125, max_cells=1000, matrices=identity_matrices, xenium_bundle_paths=xenium_bundle_paths,
                                secondary_images=imgs, secondary_matrices=xenium_to_he_matrices, draw_on_secondary=True,
                                save_func=savefn, load_func=loadfn, list_names_func=listfn)[1]

In [ ]:
if False:
    clf = dianne.loadGUIClassifier(classifierPaths, '...-gui')
else:
    # Train the classifier on the patches and the corresponding features
    body_overlap = 0.25 # Fraction overlap between the drawn annotations and the tiles
    clf, patchesCDFsMod, annotationsMod = dianne.getClassifierForFromStrokes(drawings, patchCoordinates, tile_size, body_overlap, patch_size,
                                                                            ads, samples, qs, augFunc=dianne.PCMA, alpha=0.8, seed=0, showPatches=False)
    # dianne.saveGUIClassifier(clf, classifierPaths, 'placm-chorion-15-gui', samples, ts, mpp, patch_size, tile_size, body_overlap, qs, patchesCDFsMod, annotationsMod, drawings)
    # dianne.saveGUIClassifier(clf, classifierPaths, 'placm-11-gui', samples, ts, mpp, patch_size, tile_size, body_overlap, qs, patchesCDFsMod, annotationsMod, drawings)

print(patchesCDFsMod.shape)

# Run inference and visualize the results
if True:
    if not clf is None:
        infSample = samples[0]
        multiplier = 2 # Interpolation parameter for the probability heatmap
        x, y, p = dianne.inferProbFast(ads[infSample], clf, qs, tsize=ts/mpp, R=2, erode=False)
        xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=multiplier)
        dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=infSample, invert=False)
        # viewer.set_overlay_points(xi, yi, pi, infSample, delta=tile_size / multiplier, alpha=0.5, color_low='#FFA500', color_high='#0000FF')

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ads[infSample], imgs[infSample], x, y, p, ts=ts, mpp=mpp, downfactor=16, verbose=True, savepath=classifierPaths, prefix='Amnion')
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.5, min_area=10**6, downfactor=16, sigma=100, annotation_name='Amnion', savepath=classifierPaths, prefix='Amnion')
dianne.viewContoursOnImage(imgs[infSample], geojson, fshape, level=2, figsize=(12, 12), linewidth=1)